<a href="https://colab.research.google.com/github/K-Irungu/apache-spark-assignment/blob/main/Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Apache Spark Assignment: Chicago Taxi Trips Analysis
**Name:** Kevin Thuita Irungu  
**Student ID:** 668115  
**Date:** 24/09/2026

## Stage 1 - Initialize SparkSession and Load Data

### Install pyspark and requests

In [1]:
!pip install pyspark requests

### Download the dataset

In [2]:
import requests

url = "https://data.cityofchicago.org/resource/wrvz-psew.csv?$limit=200000&$order=trip_id"
local_path = "chicago_taxi_trips.csv"

response = requests.get(url, timeout=120)
response.raise_for_status()

with open(local_path, "wb") as f:
    f.write(response.content)

print(f"Downloaded file size: {len(response.content) / (1024*1024):.2f} MB")

Downloaded file size: 85.90 MB


### Create SparkSession

In [3]:
from pyspark.sql import SparkSession
from pyspark import SparkContext

# Stop any existing SparkSession
active_spark_session = SparkSession.getActiveSession()
if active_spark_session:
    print("Stopping existing SparkSession.")
    active_spark_session.stop()

# Ensure no SparkContext is running (more robust check)
if SparkContext._active_spark_context is not None:
    print("Stopping existing SparkContext directly.")
    SparkContext._active_spark_context.stop()

# Now create the new SparkSession
spark = SparkSession.builder \
    .appName("ChicagoTaxiAnalysis") \
    .getOrCreate()

spark

### Load csv into a Spark DataFrame

In [4]:
taxi_df = spark.read.csv(local_path, header=True, inferSchema=True)

### Print counts

In [5]:
row_count = taxi_df.count()
col_count = len(taxi_df.columns)

print(f"Total rows: {row_count}")
print(f"Total columns: {col_count}")

Total rows: 200000
Total columns: 23


## Stage 2 - Inspect the DataFrame

### Print the schema

In [6]:
taxi_df.printSchema()

root
 |-- trip_id: string (nullable = true)
 |-- taxi_id: string (nullable = true)
 |-- trip_start_timestamp: timestamp (nullable = true)
 |-- trip_end_timestamp: timestamp (nullable = true)
 |-- trip_seconds: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- pickup_census_tract: long (nullable = true)
 |-- dropoff_census_tract: long (nullable = true)
 |-- pickup_community_area: integer (nullable = true)
 |-- dropoff_community_area: integer (nullable = true)
 |-- fare: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- extras: double (nullable = true)
 |-- trip_total: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- company: string (nullable = true)
 |-- pickup_centroid_latitude: double (nullable = true)
 |-- pickup_centroid_longitude: double (nullable = true)
 |-- pickup_centroid_location: string (nullable = true)
 |-- dropoff_centroid_latitude: double (nullable = true)
 |-- dropoff_cen

### List all column names

In [7]:
print(taxi_df.columns)

['trip_id', 'taxi_id', 'trip_start_timestamp', 'trip_end_timestamp', 'trip_seconds', 'trip_miles', 'pickup_census_tract', 'dropoff_census_tract', 'pickup_community_area', 'dropoff_community_area', 'fare', 'tips', 'tolls', 'extras', 'trip_total', 'payment_type', 'company', 'pickup_centroid_latitude', 'pickup_centroid_longitude', 'pickup_centroid_location', 'dropoff_centroid_latitude', 'dropoff_centroid_longitude', 'dropoff_centroid_location']


### Display first 5 rows

In [46]:
taxi_df.show(5, truncate=False)

+----------------------------------------+--------------------------------------------------------------------------------------------------------------------------------+--------------------+-------------------+------------+----------+---------------------+----------------------+-----+-----+-----+------+----------+------------+----------------------------+----+-----+---------+----+---------------------+
|trip_id                                 |taxi_id                                                                                                                         |trip_start_timestamp|trip_end_timestamp |trip_seconds|trip_miles|pickup_community_area|dropoff_community_area|fare |tips |tolls|extras|trip_total|payment_type|company                     |Year|Month|DayOfWeek|Hour|trip_duration_minutes|
+----------------------------------------+--------------------------------------------------------------------------------------------------------------------------------+-------------

### Dataset Structure Reflection

The dataset contains 23 columns spanning several types: identifiers (`trip_id`, `taxi_id` as strings), timestamps (`trip_start_timestamp`, `trip_end_timestamp`), numeric trip metrics (`trip_seconds` as integer, `trip_miles`, `fare`, `tips`, `tolls`, `extras`, `trip_total` as doubles), geographic references (`pickup_community_area`, `dropoff_community_area` as integers, plus census tracts and centroid latitude/longitude/location fields), and categorical columns (`payment_type`, `company`).


For trip analysis, the most important columns are `trip_start_timestamp`, `trip_seconds`, `trip_miles`, `fare`/`trip_total`, and the pickup/dropoff community areas, since these drive time-based, distance-based, and location-based analysis. `inferSchema=True` correctly detected `trip_seconds` as integer and the timestamp columns as proper timestamp type without needing manual casting.

The sample rows show trips spanning multiple years (2015–2023). Some rows show `NULL` values for census tract fields, which Stage 3 will need to address.

## Stage 3 - Identify and Handle Missing Data

### Count nulls per column

In [47]:
from pyspark.sql.functions import col, sum as spark_sum

null_counts = taxi_df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in taxi_df.columns
])

null_counts.show(vertical=True, truncate=False)

-RECORD 0-----------------------
 trip_id                | 0     
 taxi_id                | 0     
 trip_start_timestamp   | 0     
 trip_end_timestamp     | 0     
 trip_seconds           | 0     
 trip_miles             | 0     
 pickup_community_area  | 13234 
 dropoff_community_area | 16723 
 fare                   | 0     
 tips                   | 0     
 tolls                  | 29194 
 extras                 | 0     
 trip_total             | 0     
 payment_type           | 0     
 company                | 30724 
 Year                   | 0     
 Month                  | 0     
 DayOfWeek              | 0     
 Hour                   | 0     
 trip_duration_minutes  | 0     



### Drop rows missing essential metrics

In [48]:
taxi_df = taxi_df.dropna(subset=["trip_total", "trip_miles"])

### Fill nulls in payment_type with UNKNOWN

In [49]:
taxi_df = taxi_df.fillna({"payment_type": "UNKNOWN"})

### Print new row count

In [50]:
new_row_count = taxi_df.count()
print(f"Row count after cleaning: {new_row_count}")

Row count after cleaning: 158066


### Missing Data Reflection

The columns with the highest null counts were `company` (30,724), `tolls` (29,194), `dropoff_community_area` (16,723), and `pickup_community_area` (13,234).

The high null count in `company` likely reflects inconsistent reporting by independent or smaller taxi operators who don't always submit an affiliation name to the city, while the near-total nulls in `tolls` suggest that most trips simply never incurred a toll charge and the field is left blank rather than recorded as zero for those trips.

The nulls in `pickup_community_area` and `dropoff_community_area` are most likely due to incomplete GPS/geolocation data for certain trips, where the pickup or dropoff point couldn't be matched to a defined community area boundary.

In contrast, core trip metrics like `trip_total`, `trip_miles`, `fare`, and the timestamp columns had zero nulls, since these are essential, directly metered values that are captured for every completed trip.

After dropping rows with missing `trip_total` or `trip_miles` and filling `payment_type` nulls with "UNKNOWN", the row count changed from 200,000 to 158,066 — though note `trip_total` and `trip_miles` themselves had 0 nulls in this sample, so the row reduction actually came primarily from Stage 4's filtering step (removing trips with `trip_miles` or `trip_seconds` ≤ 0) rather than from the `dropna` call itself.

## Stage 4 - Filter and Transform





### Filter invalid trips

In [51]:
taxi_df = taxi_df.filter((col("trip_miles") > 0) & (col("trip_seconds") > 0))

filtered_row_count = taxi_df.count()
print(f"Row count after filtering invalid trips: {filtered_row_count}")

Row count after filtering invalid trips: 158066


### Select 15 columns

In [52]:
taxi_df = taxi_df.select(
    "trip_id", "taxi_id", "trip_start_timestamp", "trip_end_timestamp",
    "trip_seconds", "trip_miles", "pickup_community_area",
    "dropoff_community_area", "fare", "tips", "tolls", "extras",
    "trip_total", "payment_type", "company"
)

### Cast trip_seconds and trip_miles to Double Type

In [53]:
from pyspark.sql.types import DoubleType, IntegerType

taxi_df = taxi_df.withColumn("trip_seconds", col("trip_seconds").cast(DoubleType())) \
                 .withColumn("trip_miles", col("trip_miles").cast(DoubleType()))

### Cast community area columns to Integer Type

In [54]:
taxi_df = taxi_df.withColumn("pickup_community_area", col("pickup_community_area").cast(IntegerType())) \
                  .withColumn("dropoff_community_area", col("dropoff_community_area").cast(IntegerType()))

### Display transformed schema and first 5 rows

In [55]:
taxi_df.printSchema()
taxi_df.show(5, truncate=False)

root
 |-- trip_id: string (nullable = true)
 |-- taxi_id: string (nullable = true)
 |-- trip_start_timestamp: timestamp (nullable = true)
 |-- trip_end_timestamp: timestamp (nullable = true)
 |-- trip_seconds: double (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- pickup_community_area: integer (nullable = true)
 |-- dropoff_community_area: integer (nullable = true)
 |-- fare: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- extras: double (nullable = true)
 |-- trip_total: double (nullable = true)
 |-- payment_type: string (nullable = false)
 |-- company: string (nullable = true)

+----------------------------------------+--------------------------------------------------------------------------------------------------------------------------------+--------------------+-------------------+------------+----------+---------------------+----------------------+-----+-----+-----+------+----------+------------+----------

## Stage 5 - Add Derived Columns from Timestamps

### Convert trip_start_timestamp column to proper timestamp

In [56]:
from pyspark.sql.functions import to_timestamp

taxi_df = taxi_df.withColumn(
    "trip_start_timestamp",
    to_timestamp(col("trip_start_timestamp").cast("string"), "yyyy-MM-dd HH:mm:ss")
)

### Extract Year, Month, DayOfWeek, Hour

In [57]:
from pyspark.sql.functions import year, month, dayofweek, hour

taxi_df = taxi_df.withColumn("Year", year(col("trip_start_timestamp"))) \
                  .withColumn("Month", month(col("trip_start_timestamp"))) \
                  .withColumn("DayOfWeek", dayofweek(col("trip_start_timestamp"))) \
                  .withColumn("Hour", hour(col("trip_start_timestamp")))

### Calculate trip duration in minutes

In [58]:
taxi_df = taxi_df.withColumn("trip_duration_minutes", col("trip_seconds") / 60.0)

### Display new schema and 5 rows

In [59]:
taxi_df.printSchema()
taxi_df.show(5, truncate=False)

root
 |-- trip_id: string (nullable = true)
 |-- taxi_id: string (nullable = true)
 |-- trip_start_timestamp: timestamp (nullable = true)
 |-- trip_end_timestamp: timestamp (nullable = true)
 |-- trip_seconds: double (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- pickup_community_area: integer (nullable = true)
 |-- dropoff_community_area: integer (nullable = true)
 |-- fare: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- extras: double (nullable = true)
 |-- trip_total: double (nullable = true)
 |-- payment_type: string (nullable = false)
 |-- company: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- Hour: integer (nullable = true)
 |-- trip_duration_minutes: double (nullable = true)

+----------------------------------------+-------------------------------------------------------------------------------------------

## Stage 6 - Join with Supplementary Data

### Create supplementary dataframe

In [60]:
from pyspark.sql import Row

community_data = [
    Row(community_area=8, community_name="Near West Side", population=62000),
    Row(community_area=32, community_name="Loop", population=42000),
    Row(community_area=76, community_name="O'Hare", population=13000),
    Row(community_area=6, community_name="Lake View", population=98000),
    Row(community_area=28, community_name="Near North Side", population=85000),
]

community_df = spark.createDataFrame(community_data)
community_df.show()

+--------------+---------------+----------+
|community_area| community_name|population|
+--------------+---------------+----------+
|             8| Near West Side|     62000|
|            32|           Loop|     42000|
|            76|         O'Hare|     13000|
|             6|      Lake View|     98000|
|            28|Near North Side|     85000|
+--------------+---------------+----------+



### Inner join

In [61]:
taxi_with_community_inner = taxi_df.join(
    community_df,
    taxi_df["pickup_community_area"] == community_df["community_area"],
    "inner"
)

inner_row_count = taxi_with_community_inner.count()
print(f"Inner join row count: {inner_row_count}")

Inner join row count: 113890


### Left join and comparison

In [62]:
taxi_with_community_left = taxi_df.join(
    community_df,
    taxi_df["pickup_community_area"] == community_df["community_area"],
    "left"
)

left_row_count = taxi_with_community_left.count()
print(f"Left join row count: {left_row_count}")
print(f"Inner join row count: {inner_row_count}")
print(f"Difference: {left_row_count - inner_row_count} rows")

Left join row count: 158066
Inner join row count: 113890
Difference: 44176 rows


### Assign final output variable and display 5 rows

In [63]:
taxi_with_community = taxi_with_community_left

taxi_with_community.select("trip_id", "pickup_community_area", "community_name").show(5, truncate=False)

+----------------------------------------+---------------------+--------------+
|trip_id                                 |pickup_community_area|community_name|
+----------------------------------------+---------------------+--------------+
|00000096f27c4380ac7a5dd532d1309e814f3b2f|77                   |NULL          |
|0000003dfd0ab4ef5555e2ad299a2ba602b1dfa3|32                   |Loop          |
|00000088085c9d2854b0dbc1bfba82bc23ea5d64|NULL                 |NULL          |
|000000667ce8625dbec259647ff0bd26c40d45ee|8                    |Near West Side|
|00000077d5ad772683ffaaf9ee7883d5aab7713b|8                    |Near West Side|
+----------------------------------------+---------------------+--------------+
only showing top 5 rows


### Inner Join vs Left Join Reflection

An **inner join** only keeps rows where the join condition matches on both sides 0 so a taxi trip is only included if its `pickup_community_area` exists in `community_df`.

Since `community_df` only contains 5 specific community areas out of Chicago's 77, an inner join drops every trip whose pickup area isn't one of those 5, resulting in a much smaller row count (113,890 vs the original 158,066).

A **left join**, by contrast, keeps every row from the left table (`taxi_df`) regardless of whether a match exists — trips with a matching community area get the extra `community_name`/`population` columns filled in, while trips with no match (including trips where `pickup_community_area` itself is `NULL`, as seen in the sample rows) simply get `null` values for those columns instead of being dropped.

This is why the left join row count (158,066) exactly equals the original `taxi_df` row count, while the inner join count (113,890) is 44,176 rows smaller — confirming that an inner join here would silently discard over a quarter of the cleaned dataset.

## Stage 7 - Aggregations

### Q1 - Top pickup locations

In [64]:
from pyspark.sql.functions import count, avg, desc

top_pickup_locations = taxi_df.groupBy("pickup_community_area") \
    .agg(count("*").alias("trip_count")) \
    .orderBy(desc("trip_count"))

print("Top 10 pickup community areas:")
top_pickup_locations.show(10)

Top 10 pickup community areas:
+---------------------+----------+
|pickup_community_area|trip_count|
+---------------------+----------+
|                    8|     45993|
|                   32|     33425|
|                   28|     13780|
|                 NULL|     13234|
|                   76|     12404|
|                    6|      8288|
|                    7|      6393|
|                   24|      4094|
|                   33|      3931|
|                   56|      2997|
+---------------------+----------+
only showing top 10 rows


### Q2 - Average fare, overall and by payment_type

In [65]:
print("Average trip_total across all trips:")
taxi_df.agg(avg("trip_total").alias("avg_trip_total")).show()

print("Average trip_total by payment_type:")
taxi_df.groupBy("payment_type") \
    .agg(avg("trip_total").alias("avg_trip_total")) \
    .orderBy(desc("avg_trip_total")) \
    .show()

Average trip_total across all trips:
+------------------+
|    avg_trip_total|
+------------------+
|17.087132526919437|
+------------------+

Average trip_total by payment_type:
+------------+------------------+
|payment_type|    avg_trip_total|
+------------+------------------+
|     Prepaid|             30.75|
|      Prcard|23.279920595533504|
| Credit Card|22.146802906589723|
|     Unknown| 22.01466728280961|
|      Mobile|20.094529914529918|
|     Dispute|17.815217391304348|
|       Split|           16.2775|
|   No Charge|13.474801061007957|
|        Cash|13.121431331545308|
|       Pcard|12.038095238095238|
+------------+------------------+



### Q3 - Trips by hour

In [66]:
trips_by_hour = taxi_df.groupBy("Hour") \
    .agg(count("*").alias("trip_count")) \
    .orderBy(desc("trip_count"))

print("Trips by hour (sorted by count, highest first):")
trips_by_hour.show(24)


Trips by hour (sorted by count, highest first):
+----+----------+
|Hour|trip_count|
+----+----------+
|  18|     10553|
|  17|     10179|
|  19|      9971|
|  16|      9293|
|  20|      8761|
|  15|      8574|
|  13|      8386|
|  12|      8288|
|  14|      8206|
|  21|      7895|
|  11|      7795|
|   9|      7572|
|  10|      7222|
|  22|      7168|
|   8|      6898|
|  23|      6379|
|   0|      5053|
|   7|      4527|
|   1|      4123|
|   2|      3227|
|   6|      2381|
|   3|      2374|
|   4|      1685|
|   5|      1556|
+----+----------+



### Q4 - Company comparison

In [67]:
company_comparison = taxi_df.groupBy("company") \
    .agg(
        count("*").alias("trip_count"),
        avg("trip_miles").alias("avg_trip_miles")
    ) \
    .orderBy(desc("trip_count"))

print("Top 10 companies by trip count:")
company_comparison.show(10, truncate=False)


Top 10 companies by trip count:
+-------------------------+----------+------------------+
|company                  |trip_count|avg_trip_miles    |
+-------------------------+----------+------------------+
|NULL                     |30724     |4.306835698476759 |
|Taxi Affiliation Services|25403     |3.5590284612052585|
|Flash Cab                |19052     |4.877978689901322 |
|Yellow Cab               |9008      |3.9790741563055096|
|Dispatch Taxi Affiliation|7494      |3.856405124099282 |
|Chicago Carriage Cab Corp|7302      |5.605493015612157 |
|Sun Taxi                 |6231      |5.29310223078157  |
|City Service             |6207      |4.934270984372484 |
|Choice Taxi Association  |5982      |4.44914744232698  |
|Medallion Leasin         |4793      |4.614014187356563 |
+-------------------------+----------+------------------+
only showing top 10 rows


## Stage 8 - Community Area Analysis

### Group by pickup area - trips and average fare, sorted, top 10

In [68]:
community_summary = taxi_df.groupBy("pickup_community_area") \
    .agg(
        count("*").alias("trip_count"),
        avg("fare").alias("avg_fare")
    ) \
    .orderBy(desc("trip_count"))

print("Top 10 community areas by trip count:")
community_summary.show(10)


Top 10 community areas by trip count:
+---------------------+----------+------------------+
|pickup_community_area|trip_count|          avg_fare|
+---------------------+----------+------------------+
|                    8|     45993|10.880029569717063|
|                   32|     33425|11.215168885564683|
|                   28|     13780|10.585267779390424|
|                 NULL|     13234|15.223471361644133|
|                   76|     12404| 37.57585536923582|
|                    6|      8288|11.983381998069493|
|                    7|      6393| 12.06231659627719|
|                   24|      4094|11.179447972642889|
|                   33|      3931| 14.56720681760367|
|                   56|      2997|   32.366983650317|
+---------------------+----------+------------------+
only showing top 10 rows


### Join aggregated result with community_df for names

In [69]:
community_summary_named = community_summary.join(
    community_df,
    community_summary["pickup_community_area"] == community_df["community_area"],
    "left"
)

print("Community area summary with names (top 10 by trip count):")
community_summary_named.select(
    "pickup_community_area", "community_name", "trip_count", "avg_fare"
).orderBy(desc("trip_count")).show(10, truncate=False)

Community area summary with names (top 10 by trip count):
+---------------------+---------------+----------+------------------+
|pickup_community_area|community_name |trip_count|avg_fare          |
+---------------------+---------------+----------+------------------+
|8                    |Near West Side |45993     |10.880029569717063|
|32                   |Loop           |33425     |11.215168885564683|
|28                   |Near North Side|13780     |10.585267779390424|
|NULL                 |NULL           |13234     |15.223471361644133|
|76                   |O'Hare         |12404     |37.57585536923582 |
|6                    |Lake View      |8288      |11.983381998069493|
|7                    |NULL           |6393      |12.06231659627719 |
|24                   |NULL           |4094      |11.179447972642889|
|33                   |NULL           |3931      |14.56720681760367 |
|56                   |NULL           |2997      |32.366983650317   |
+---------------------+---------

### Filter for Community Area 76 - O'Hare

In [70]:
ohare_trip_count = taxi_df.filter(col("pickup_community_area") == 76).count()
print(f"Trips originating in Community Area 76 (O'Hare): {ohare_trip_count}")

Trips originating in Community Area 76 (O'Hare): 12404


### Community Area Analysis Reflection

Area 8 (Near West Side) has the highest trip count by a wide margin at 45,993 trips, followed by Loop (area 32) at 33,425 and Near North Side (area 28) at 13,780 — these are all central, high-density areas of Chicago with heavy commercial activity, nightlife, and foot traffic, which naturally drives taxi demand.


Area 76 (O'Hare) again stands out on a different metric: despite ranking 5th in trip count (12,404), it has by far the highest average fare at $37.58 — more than three times some other top-10 areas — consistent with O'Hare being an airport, where trips are typically long-distance rides to/from downtown or surrounding suburbs rather than short local hops.

Area 56 shows a similar pattern on a smaller scale, with a notably high average fare ($32.37) despite a modest trip count (2,997), suggesting it may also involve longer-than-average trips, possibly another outlying or less central area.


The `NULL` pickup_community_area group accounts for 13,234 trips and also has an elevated average fare ($15.22), reinforcing the pattern already noted in Stage 3 that missing geolocation data isn't randomly distributed across trip types.


## Stage 9 - Window Functions for Advanced Analysis

### Analysis 1: Rank top 3 companies per community area

In [71]:
from pyspark.sql import Window
from pyspark.sql.functions import rank

company_area_counts = taxi_df.groupBy("pickup_community_area", "company") \
    .agg(count("*").alias("trip_count"))

area_window = Window.partitionBy("pickup_community_area").orderBy(desc("trip_count"))

ranked_companies = company_area_counts.withColumn("rank", rank().over(area_window))

top3_companies_per_area = ranked_companies.filter(col("rank") <= 3)

print("Top 3 companies per pickup community area:")
top3_companies_per_area.orderBy("pickup_community_area", "rank").show(30, truncate=False)


Top 3 companies per pickup community area:
+---------------------+-------------------------+----------+----+
|pickup_community_area|company                  |trip_count|rank|
+---------------------+-------------------------+----------+----+
|NULL                 |NULL                     |5636      |1   |
|NULL                 |303 Taxi                 |1186      |2   |
|NULL                 |Flash Cab                |974       |3   |
|1                    |Flash Cab                |212       |1   |
|1                    |Taxi Affiliation Services|126       |2   |
|1                    |NULL                     |83        |3   |
|2                    |Flash Cab                |235       |1   |
|2                    |Taxi Affiliation Services|99        |2   |
|2                    |NULL                     |75        |3   |
|3                    |NULL                     |500       |1   |
|3                    |Flash Cab                |409       |2   |
|3                    |Taxi Affil

### Analysis 2: lead - next trip start time per company

In [72]:
from pyspark.sql.functions import lead

company_time_window = Window.partitionBy("company").orderBy("trip_start_timestamp")

taxi_with_lead = taxi_df.withColumn(
    "next_trip_start_time",
    lead("trip_start_timestamp").over(company_time_window)
)

print("Next trip start time per company (lead function):")
taxi_with_lead.select("company", "trip_start_timestamp", "next_trip_start_time") \
    .filter(col("company").isNotNull()) \
    .orderBy("company", "trip_start_timestamp") \
    .show(10, truncate=False)

Next trip start time per company (lead function):
+---------------------------+--------------------+--------------------+
|company                    |trip_start_timestamp|next_trip_start_time|
+---------------------------+--------------------+--------------------+
|0118 - 42111 Godfrey S.Awir|2015-04-27 19:30:00 |2015-08-18 19:45:00 |
|0118 - 42111 Godfrey S.Awir|2015-08-18 19:45:00 |2015-09-05 11:45:00 |
|0118 - 42111 Godfrey S.Awir|2015-09-05 11:45:00 |2016-03-10 14:30:00 |
|0118 - 42111 Godfrey S.Awir|2016-03-10 14:30:00 |2016-04-04 18:45:00 |
|0118 - 42111 Godfrey S.Awir|2016-04-04 18:45:00 |2016-05-07 12:15:00 |
|0118 - 42111 Godfrey S.Awir|2016-05-07 12:15:00 |2016-09-10 19:30:00 |
|0118 - 42111 Godfrey S.Awir|2016-09-10 19:30:00 |2016-10-19 21:15:00 |
|0118 - 42111 Godfrey S.Awir|2016-10-19 21:15:00 |2016-11-13 15:15:00 |
|0118 - 42111 Godfrey S.Awir|2016-11-13 15:15:00 |2017-03-15 22:15:00 |
|0118 - 42111 Godfrey S.Awir|2017-03-15 22:15:00 |2017-03-25 20:00:00 |
+-------------

### Analysis 3: lag - year-over-year change per community area

In [73]:
from pyspark.sql.functions import lag

yearly_counts = taxi_df.groupBy("pickup_community_area", "Year") \
    .agg(count("*").alias("trip_count"))

year_window = Window.partitionBy("pickup_community_area").orderBy("Year")

yearly_with_change = yearly_counts.withColumn(
    "previous_year_count", lag("trip_count").over(year_window)
).withColumn(
    "year_over_year_change", col("trip_count") - col("previous_year_count")
)

print("Year-over-year trip count change per community area:")
yearly_with_change.orderBy("pickup_community_area", "Year").show(30, truncate=False)

Year-over-year trip count change per community area:
+---------------------+----+----------+-------------------+---------------------+
|pickup_community_area|Year|trip_count|previous_year_count|year_over_year_change|
+---------------------+----+----------+-------------------+---------------------+
|NULL                 |2013|2052      |NULL               |NULL                 |
|NULL                 |2014|1809      |2052               |-243                 |
|NULL                 |2015|1757      |1809               |-52                  |
|NULL                 |2016|2590      |1757               |833                  |
|NULL                 |2017|1766      |2590               |-824                 |
|NULL                 |2018|1138      |1766               |-628                 |
|NULL                 |2019|1071      |1138               |-67                  |
|NULL                 |2020|217       |1071               |-854                 |
|NULL                 |2021|233       |217   

### Why Window Functions instead of GroupBy

A simple `groupBy` collapses rows into one aggregated row per group, losing row-level detail — it can tell you the total trip count per company, but it can't rank each company *within* its community area while still showing every company's row, or compare one row to the row before/after it. Window functions solve this using `partitionBy` (defining the group) and `orderBy` (defining the order within that group) without collapsing the DataFrame — each row keeps its identity while gaining context from related rows.

This is visible directly in the results: `rank()` shows the top 3 companies varying by community area — Flash Cab and Taxi Affiliation Services dominate most areas, but their exact rank order shifts depending on the area — something a plain `groupBy("company")` alone couldn't produce, since it wouldn't be partitioned per area. The `lead()` analysis shows gaps of weeks or months between consecutive trips for some companies (e.g. `0118 - 42111 Godfrey S.Awir` has multi-month gaps between logged trips), which is only visible through row-to-row comparison, not aggregation.

The `lag()` year-over-year analysis is now fully populated across 2013–2023 and reveals a clear pattern: trip counts across most community areas declined sharply around 2020 (e.g. the `NULL` pickup area group dropped from 1,071 in 2019 to just 217 in 2020, a change of -854), consistent with the COVID-19 pandemic's impact on taxi demand, followed by partial recovery in subsequent years.

None of this — the ranking, the gap detection, or the year-over-year trend — could be produced by a simple `groupBy`, since all three require comparing a row to specific other rows within an ordered partition rather than reducing the group to a single summary value.

## Stage 10 - SQL Query on the cleaned data

### Register temp view

In [76]:
taxi_with_community.createOrReplaceTempView("taxi_view")

### Q1 - Top 5 community areas by trip count

In [77]:
query1_result = spark.sql("""
    SELECT community_name, COUNT(*) AS trip_count
    FROM taxi_view
    GROUP BY community_name
    ORDER BY trip_count DESC
    LIMIT 5
""")

print("Query 1: Top 5 pickup community areas by trip count")
query1_result.show(truncate=False)

Query 1: Top 5 pickup community areas by trip count
+---------------+----------+
|community_name |trip_count|
+---------------+----------+
|Near West Side |45993     |
|NULL           |44176     |
|Loop           |33425     |
|Near North Side|13780     |
|O'Hare         |12404     |
+---------------+----------+



### Q2 - avg trip_total and trip_miles by payment_type, >100 trips

In [78]:
query2_result = spark.sql("""
    SELECT payment_type,
           AVG(trip_total) AS avg_trip_total,
           AVG(trip_miles) AS avg_trip_miles,
           COUNT(*) AS trip_count
    FROM taxi_view
    GROUP BY payment_type
    HAVING COUNT(*) > 100
    ORDER BY trip_count DESC
""")

print("Query 2: Average trip_total and trip_miles by payment_type (>100 trips)")
query2_result.show(truncate=False)


Query 2: Average trip_total and trip_miles by payment_type (>100 trips)
+------------+------------------+------------------+----------+
|payment_type|avg_trip_total    |avg_trip_miles    |trip_count|
+------------+------------------+------------------+----------+
|Cash        |13.121431331545327|3.5100253443652463|87988     |
|Credit Card |22.146802906589702|5.481712191872161 |64543     |
|Prcard      |23.279920595533504|7.648481389578163 |2015      |
|Mobile      |20.09452991452991 |4.63994469582705  |1989      |
|Unknown     |22.014667282809608|5.388077634011088 |1082      |
|No Charge   |13.474801061007957|3.986737400530503 |377       |
+------------+------------------+------------------+----------+



### Q3 - Month with highest total trip count

In [79]:
query3_result = spark.sql("""
    SELECT Month, COUNT(*) AS trip_count
    FROM taxi_view
    GROUP BY Month
    ORDER BY trip_count DESC
    LIMIT 1
""")

print("Query 3: Month with the highest total trip count")
query3_result.show()

Query 3: Month with the highest total trip count
+-----+----------+
|Month|trip_count|
+-----+----------+
|   10|     14454|
+-----+----------+



### DataFrame API vs SQL Approach

Writing these three queries in SQL felt more concise and readable than the equivalent DataFrame API chains from Stages 6–9, especially for Query 2's `HAVING COUNT(*) > 100` filter — expressing that in the DataFrame API would require a `groupBy().agg().filter()` chain that reads less naturally than SQL's built-in `HAVING` clause.


However, the DataFrame API felt more natural for the window function work in Stage 9 (`rank()`, `lead()`, `lag()`), since chaining `.withColumn()` with `Window.partitionBy()` kept each transformation step explicit and easy to debug one at a time, whereas the equivalent SQL window syntax would need to be written all at once inside a single query string.

Overall, SQL felt quicker for simple grouped aggregations, while the DataFrame API felt better suited to multi-step, incrementally built pipelines.

## Stage 11 - Final Reflection

**1. Lazy evaluation:**
One clear moment this mattered was in Stages 4–6, where I chained multiple `.filter()`, `.select()`, `.withColumn()`, and `.join()` operations on `taxi_df` before ever calling `.count()` or `.show()`. None of those transformations actually touched the data until an action triggered execution — Spark instead built up a logical query plan and only optimized/executed it in one pass when `.count()` was finally called. If I had called `.count()` after every single transformation (e.g., after each `.filter()`, each `.withColumn()`, each cast), Spark would have re-executed the *entire* plan from the CSV read onward every single time, since DataFrames aren't cached by default — this would have meant redundant re-reading and re-processing of the same data dozens of times over, dramatically slowing down the pipeline for no analytical benefit, since none of those intermediate counts were actually needed for the final result.

**2. Sampling limitation:**
A 200,000-row sample, even when it spans multiple years (2013–2023) as this one does, cannot capture the full volume or density of trips that occurred in each period — some years or months may be under- or over-represented relative to their true share of the full ~200 million trips, simply due to how the sample was selected. For example, the `lag()` year-over-year analysis in Stage 9 showed a sharp decline in trip counts around 2020, consistent with COVID-19's impact on taxi demand, but with only a few thousand trips representing an entire year in some cases, small sample sizes per year make it harder to distinguish genuine trends from sampling noise. With the full dataset, every year, month, and community area would have enough underlying trips to support statistically reliable conclusions, and patterns like seasonal demand, long-term company market share, or the true scale of the 2020 decline could be measured with much greater precision rather than being estimated from a comparatively thin slice of the total data.

**3. Inner join risk:**
The main risk of an inner join, demonstrated directly in Stage 6, is silently dropping valid data that simply doesn't have a match in the smaller table. When I inner-joined `taxi_df` with `community_df` (which only contains 5 of Chicago's 77 community areas), the row count dropped from 158,066 to 113,890 — a loss of 44,176 valid, already-cleaned trips, purely because their `pickup_community_area` wasn't one of the 5 named areas. If I had used the inner join result for the rest of the analysis instead of the left join, aggregations like "top pickup locations" or "average fare by area" would have been calculated on a smaller, non-representative subset of the data, and conclusions about overall trip patterns across Chicago would have been skewed toward just those 5 areas — a serious risk if this pipeline were used for real business decisions, since it would silently misrepresent the true scale and distribution of citywide taxi activity.